In [ ]:
"""Parte 1 — tratamento das bases e construcao do painel fundamentalista.

Versao brasileira adaptada: 97 formulas implementadas, 76 variaveis com dados
na execucao de 27/10/2026 e 21 sem entradas suficientes. Nao e replica integral.

Coloque este .py junto das bases Excel na raiz do repositorio Projeto-Quant.
Uso: python Parte1_construcao_painel_fundamentalista.py
Saida: resultados_parte1/. As bases de entrada permanecem intactas.
Requer: pandas, numpy, openpyxl e pyarrow. Nao acessa internet.

Inclui leitores, calendario, formulas, coleta complementar e relatorios
neste unico arquivo; nao depende dos leitores antigos do repositorio.
"""



"""
Parte2_treino.py  (v2.0)

Dois experimentos, mantidos separados:
  --modo cv        : reproducao do desenho do paper (validacao cruzada TRANSVERSAL dentro de
                     cada mes, 5 blocos, agrupados por emissor). Mede qualidade de precificacao
                     relativa, NAO e um teste negociavel.
  --modo temporal  : janela expansiva/rolante; treina so com meses anteriores ao mes previsto.
                     Experimento a usar se o objetivo for algo proximo de uma estrategia.

Cada configuracao (parametros, features, seed, modo, hash do painel, versao) gera um diretorio
proprio: resultados/run_<hash>. Mudou a metodologia -> novo hash -> nada de resultado antigo reaproveitado.
"""
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")          # evita competicao de CPU entre joblib e BLAS

import argparse
import hashlib
import json
import re
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from joblib import Parallel, delayed

VERSAO_CODIGO = "2.0"
SEED = 42
ALVOS = ["lnm2b", "lnv2a", "lnv2s"]
ALVOS_EV = ["lnv2a", "lnv2s"]
DENOM = {"lnm2b": "pl", "lnv2a": "ativos", "lnv2s": "vendas"}

BASE = Path(__file__).parent
ARQ_PAINEL = BASE / "painel_modelagem.parquet"
ARQ_FEATURES = BASE / "features_permitidas.json"   # opcional: lista explicita e congelada
SAIDA = BASE / "resultados"

MIN_OBS = 60            # minimo de observacoes validas (por alvo) no mes no modo cv
N_FOLDS = 5
MIN_FOLD = 5            # minimo de linhas em treino/validacao/teste de cada dobra
JANELA_TEMP = 36        # meses de historico no modo temporal
N_VAL_MESES = 3         # ultimos meses do historico usados como validacao (early stopping)
MIN_TREINO_TEMP = 300
# NOTA: confirmar contra a secao de metodos do paper antes de chamar de "paper".
PARAMS_PAPER = dict(objective="regression", learning_rate=0.1, num_leaves=31,
                    min_data_in_leaf=20, num_threads=1, verbose=-1,
                    seed=SEED, deterministic=True, force_row_wise=True)
# Decisao metodologica para amostra pequena (documentar; nao ha evidencia de superioridade).
PARAMS_PEQUENA = {**PARAMS_PAPER, "num_leaves": 8, "min_data_in_leaf": 5}

NAO_FEATURES = {"ticker", "data", "valor_mercado", "ev", "bloco", "emissor", *ALVOS}
PADROES_PROIBIDOS = re.compile(
    r"(preco|price|cotacao|retorno_|return|mom|mkt|mcap|valor_mercado|^ev($|_)|m2b|v2a|v2s"
    r"|target|alvo|bloco|emissor|^data|date)", re.I)

# ----------------------------------------------------------------------------- utilidades
def grava_parquet(df, arq):
    """Escrita atomica: arquivo incompleto de execucao interrompida nunca vira resultado."""
    tmp = arq.with_suffix(".tmp")
    df.to_parquet(tmp)
    os.replace(tmp, arq)

def pred_valido(arq):
    if not arq.exists():
        return False
    try:
        pd.read_parquet(arq, columns=["ticker"])
        return True
    except Exception:
        arq.unlink(missing_ok=True)
        return False

def hash_arquivo(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

# ----------------------------------------------------------------------------- features
def selecionar_features(colunas, arq_json=ARQ_FEATURES):
    """Lista explicita (features_permitidas.json) se existir; senao, candidatas com bloqueio por nome."""
    if Path(arq_json).exists():
        feats = json.loads(Path(arq_json).read_text(encoding="utf-8"))
        ausentes = [c for c in feats if c not in colunas]
        proibidas = [c for c in feats if c in NAO_FEATURES]
        if ausentes or proibidas:
            raise ValueError(f"features_permitidas.json invalido. Ausentes: {ausentes} | proibidas: {proibidas}")
        return list(feats)
    cand = [c for c in colunas if c not in NAO_FEATURES]
    suspeitas = [c for c in cand if PADROES_PROIBIDOS.search(c)]
    if suspeitas:
        raise ValueError(
            f"Colunas com nome suspeito de preco/alvo/data: {suspeitas}. Se forem legitimas, congele a lista "
            f"em {ARQ_FEATURES.name}; se nao, remova-as do painel.")
    return cand

def validar_features(df, features, limite_corr=0.98):
    for c in features:
        if not (pd.api.types.is_numeric_dtype(df[c]) or isinstance(df[c].dtype, pd.CategoricalDtype)):
            raise TypeError(f"Feature '{c}' tem dtype {df[c].dtype}; so numerico ou category.")
    num = [c for c in features if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c])]
    if num and np.isinf(df[num].to_numpy(dtype=float)).any():
        raise ValueError("Features com +/-inf: trate no painel (ex.: razoes com denominador zero).")
    for a in ALVOS:
        y = df[a].replace([np.inf, -np.inf], np.nan)
        corr = df[num].corrwith(y, method="spearman").abs()
        if corr.max() > limite_corr:
            raise ValueError(f"Possivel vazamento: '{corr.idxmax()}' tem |Spearman|={corr.max():.4f} com {a}.")

# ----------------------------------------------------------------------------- emissor / blocos
def chave_emissor(df):
    """Coluna 'emissor' se existir. Fallback: 4 letras iniciais do ticker (PETR3/PETR4 -> PETR).
    O fallback e HEURISTICA (units, tickers reaproveitados): valide com o cadastro da Economatica."""
    if "emissor" in df.columns:
        return df["emissor"].astype(str)
    t = df["ticker"].astype(str)
    return t.str.extract(r"^([A-Za-z]{4})")[0].fillna(t)

def atribui_blocos(emissor, n_folds, seed):
    """Todas as linhas de um mesmo emissor caem no mesmo bloco; blocos balanceados por n de linhas."""
    rng = np.random.default_rng(seed)
    tam = emissor.value_counts()
    ordem = list(tam.index[rng.permutation(len(tam))])
    ordem.sort(key=lambda g: -tam[g])                       # estavel
    carga = np.zeros(n_folds, dtype=int)
    mapa = {}
    for g in ordem:
        f = int(np.argmin(carga))
        mapa[g] = f
        carga[f] += tam[g]
    return emissor.map(mapa).astype(int)

# ----------------------------------------------------------------------------- alvo valido
def filtra_alvo(df, alvo):
    """Alvo finito, denominador e valor de mercado > 0; EV finito para alvos de EV."""
    ok = np.isfinite(df[alvo].to_numpy(dtype=float))
    ok &= df[DENOM[alvo]].to_numpy(dtype=float) > 0
    ok &= df["valor_mercado"].to_numpy(dtype=float) > 0
    if alvo in ALVOS_EV:
        ok &= np.isfinite(df["ev"].to_numpy(dtype=float))
    return df.loc[ok]

# ----------------------------------------------------------------------------- modelo
def treina_e_preve(treino, val, teste, alvo, features, params):
    dtr = lgb.Dataset(treino[features], treino[alvo])        # category do pandas e detectada
    dva = lgb.Dataset(val[features], val[alvo], reference=dtr)
    modelo = lgb.train(params, dtr, num_boost_round=2000, valid_sets=[dva],
                       callbacks=[lgb.early_stopping(50, verbose=False)])
    n_arv = modelo.best_iteration if modelo.best_iteration and modelo.best_iteration > 0 \
        else modelo.current_iteration()
    pred_val = modelo.predict(val[features], num_iteration=n_arv)
    smear = float(np.mean(np.exp(val[alvo].to_numpy(dtype=float) - pred_val)))   # Duan (1983)

    p = teste[["ticker", "data", "valor_mercado", "ev", DENOM[alvo], alvo]].rename(
        columns={DENOM[alvo]: "denominador", alvo: "log_real"}).copy()
    p["alvo"] = alvo
    p["pred_log"] = modelo.predict(teste[features], num_iteration=n_arv)
    p["n_arvores"] = n_arv
    p["n_treino"] = len(treino)
    p["smear"] = smear
    g = modelo.feature_importance("gain")
    imp = pd.DataFrame({"feature": modelo.feature_name(), "ganho_rel": g / (g.sum() or 1.0), "alvo": alvo})
    return p, modelo, imp


def roda_mes_cv(df_mes, features, mes, params, run_dir, salvar_modelos):
    arq = run_dir / f"pred_{mes:%Y%m}.parquet"
    if pred_valido(arq):
        return "reutilizado"
    t0 = time.time()
    df = df_mes.sort_values("ticker").reset_index(drop=True)
    df["emissor"] = chave_emissor(df)
    if df["emissor"].nunique() < N_FOLDS:
        return "poucos_emissores"
    df["bloco"] = atribui_blocos(df["emissor"], N_FOLDS, SEED + int(mes.strftime("%Y%m")))
    grava_parquet(df[["ticker", "data", "emissor", "bloco"]], run_dir / f"blocos_{mes:%Y%m}.parquet")

    saidas, imps = [], []
    for alvo in ALVOS:
        d = filtra_alvo(df, alvo)
        if len(d) < MIN_OBS:
            continue
        for k in range(N_FOLDS):
            teste = d[d.bloco == k]
            val = d[d.bloco == (k + 1) % N_FOLDS]
            treino = d[~d.bloco.isin([k, (k + 1) % N_FOLDS])]
            if min(len(teste), len(val), len(treino)) < MIN_FOLD:
                continue
            p, modelo, imp = treina_e_preve(treino, val, teste, alvo, features, params)
            p["bloco"] = k
            saidas.append(p)
            imp["bloco"] = k
            imps.append(imp)
            if salvar_modelos:
                modelo.save_model(str(run_dir / "modelos" / f"modelo_{mes:%Y%m}_{alvo}_{k}.txt"))
    if not saidas:
        return "sem_previsoes"
    out = pd.concat(saidas, ignore_index=True)
    out["segundos_mes"] = time.time() - t0
    grava_parquet(pd.concat(imps, ignore_index=True).assign(data=mes), run_dir / f"imp_{mes:%Y%m}.parquet")
    grava_parquet(out, arq)
    return "ok"
def roda_mes_temporal(janela, features, mes, params, run_dir, salvar_modelos):
    """Preve o mes `mes` usando apenas meses estritamente anteriores (features e alvos ja observaveis)."""
    arq = run_dir / f"pred_{mes:%Y%m}.parquet"
    if pred_valido(arq):
        return "reutilizado"
    t0 = time.time()
    hist = janela[janela.data < mes]
    meses_h = np.sort(hist.data.unique())
    if len(meses_h) < N_VAL_MESES + 6:
        return "historico_curto"
    corte = meses_h[-N_VAL_MESES]
    saidas, imps = [], []
    for alvo in ALVOS:
        d = filtra_alvo(hist, alvo)
        treino, val = d[d.data < corte], d[d.data >= corte]
        teste = filtra_alvo(janela[janela.data == mes], alvo)
        if len(treino) < MIN_TREINO_TEMP or min(len(val), len(teste)) < MIN_FOLD:
            continue
        p, modelo, imp = treina_e_preve(treino, val, teste, alvo, features, params)
        p["bloco"] = -1
        saidas.append(p)
        imps.append(imp.assign(bloco=-1))
        if salvar_modelos:
            modelo.save_model(str(run_dir / "modelos" / f"modelo_{mes:%Y%m}_{alvo}.txt"))
    if not saidas:
        return "sem_previsoes"
    out = pd.concat(saidas, ignore_index=True)
    out["segundos_mes"] = time.time() - t0
    grava_parquet(pd.concat(imps, ignore_index=True).assign(data=mes), run_dir / f"imp_{mes:%Y%m}.parquet")
    grava_parquet(out, arq)
    return "ok"
# ----------------------------------------------------------------------------- conversao e metricas
def converte_e_erro(p):
    """
    Identidades (EV = M + DL, logo DL = EV - M):
      lnm2b: M_prev = exp(pred) * PL
      lnv2*: EV_prev = exp(pred) * denominador ; M_prev = EV_prev - (EV - M)
    exp(pred_log) e o multiplo TIPICO (estimativa de mediana/media geometrica condicional), nao o valor
    esperado em niveis. Versao *_smear aplica o fator de Duan (media de exp(residuo) na validacao)
    como analise de sensibilidade.
    """
    p = p.reset_index(drop=True).copy()
    eh_ev = p["alvo"].isin(ALVOS_EV).to_numpy()
    vm = p["valor_mercado"].to_numpy(float)
    den = p["denominador"].to_numpy(float)
    dl = (p["ev"] - p["valor_mercado"]).to_numpy(float)
    # checagem da identidade usando o proprio valor real
    recon = np.exp(p["log_real"].to_numpy(float)) * den - np.where(eh_ev, dl, 0.0)
    p["desvio_identidade_rel"] = np.abs(recon / vm - 1)

    p["multiplo_prev"] = np.exp(p["pred_log"].to_numpy(float))
    for suf, fator in (("", 1.0), ("_smear", p["smear"].to_numpy(float))):
        nivel = p["multiplo_prev"].to_numpy(float) * fator * den
        vm_prev = np.where(eh_ev, nivel - dl, nivel)
        p[f"valor_mercado_prev{suf}"] = vm_prev
        p[f"erro_pct_vm{suf}"] = vm_prev / vm - 1
    p["vm_prev_nao_positivo"] = p["valor_mercado_prev"] <= 0
    p["erro_log_multiplo"] = p["pred_log"] - p["log_real"]
    vp = np.where(p["valor_mercado_prev"] > 0, p["valor_mercado_prev"], np.nan)
    p["erro_log_vm"] = np.log(vp / vm)
    return p

def resumo(p, chaves):
    q = p.assign(abs_erro_pct_vm=p["erro_pct_vm"].abs(), abs_erro_log=p["erro_log_multiplo"].abs(),
                 sq_erro_log=p["erro_log_multiplo"] ** 2, abs_erro_log_vm=p["erro_log_vm"].abs())
    return q.groupby(chaves).agg(
        n_previsoes=("ticker", "size"),
        n_empresas=("ticker", "nunique"),
        vies_mediano_pct_vm=("erro_pct_vm", "median"),
        mediana_abs_pct_vm=("abs_erro_pct_vm", "median"),
        mae_log_multiplo=("abs_erro_log", "mean"),
        rmse_log_multiplo=("sq_erro_log", lambda s: float(np.sqrt(s.mean()))),
        mediana_erro_log_vm=("erro_log_vm", "median"),
        mae_log_vm=("abs_erro_log_vm", "mean"),
        p05_erro_log_multiplo=("erro_log_multiplo", lambda s: s.quantile(0.05)),
        p95_erro_log_multiplo=("erro_log_multiplo", lambda s: s.quantile(0.95)),
        frac_vm_prev_nao_positivo=("vm_prev_nao_positivo", "mean"),
        vies_mediano_pct_vm_smear=("erro_pct_vm_smear", "median"),
        n_arvores_medio=("n_arvores", "mean"),
    )

def consolida(run_dir):
    arqs = sorted(run_dir.glob("pred_*.parquet"))
    if not arqs:
        print("Nenhuma previsao gerada: nada a consolidar.")
        return None
    p = pd.concat([pd.read_parquet(f) for f in arqs], ignore_index=True)
    dup = int(p.duplicated(["ticker", "data", "alvo"]).sum())
    if dup:
        raise RuntimeError(f"{dup} previsoes duplicadas (ticker, data, alvo).")
    p = converte_e_erro(p)
    p["ano"] = p["data"].dt.year
    p["ano_mes"] = p["data"].dt.strftime("%Y-%m")
    desv = p.loc[p["alvo"].isin(ALVOS_EV), "desvio_identidade_rel"].max()
    if np.isfinite(desv) and desv > 1e-6:
        warnings.warn(f"Identidade EV = M + (EV-M) falha (desvio relativo max {desv:.2e}).")
    grava_parquet(p, run_dir / "previsoes_oos.parquet")
    r_alvo = resumo(p, ["alvo"])
    r_alvo.to_csv(run_dir / "metricas_por_alvo.csv")
    resumo(p, ["alvo", "ano"]).to_csv(run_dir / "metricas_por_ano.csv")
    resumo(p, ["alvo", "ano_mes"]).to_csv(run_dir / "metricas_por_mes.csv")
    imps = [pd.read_parquet(f) for f in sorted(run_dir.glob("imp_*.parquet"))]
    if imps:
        pd.concat(imps).groupby(["alvo", "feature"])["ganho_rel"].mean().unstack(0).to_csv(
            run_dir / "importancia_media.csv")
    with pd.option_context("display.width", 200, "display.max_columns", 30):
        print(r_alvo.T)
    return p
# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--modo", choices=["cv", "temporal"], default="cv")
    ap.add_argument("--params", choices=["paper", "pequena"], default="pequena")
    ap.add_argument("--n-meses", type=int, default=12, help="0 = todos os meses elegiveis")
    ap.add_argument("--jobs", type=int, default=max(1, (os.cpu_count() or 2) - 1))
    ap.add_argument("--sem-modelos", action="store_true")
    a = ap.parse_args()

    painel = pd.read_parquet(ARQ_PAINEL)
    obrig = ["ticker", "data", "valor_mercado", "ev", *DENOM.values(), *ALVOS]
    falt = [c for c in obrig if c not in painel.columns]
    if falt:
        raise KeyError(f"Colunas ausentes no painel: {falt}")
    painel["data"] = pd.to_datetime(painel["data"])
    if painel.duplicated(["ticker", "data"]).any():
        raise ValueError("Painel com ticker-data duplicado: corrija na Parte1.")
    if "setor" in painel.columns:
        painel["setor"] = painel["setor"].astype("category")
    features = selecionar_features(list(painel.columns))
    validar_features(painel, features)
    print("features:", features)
    if "setor" not in features:
        print("AVISO: 'setor' nao esta entre as features.")
    params = PARAMS_PAPER if a.params == "paper" else PARAMS_PEQUENA
    config = dict(versao=VERSAO_CODIGO, modo=a.modo, params=params, seed=SEED, min_obs=MIN_OBS,
                  n_folds=N_FOLDS, min_fold=MIN_FOLD, janela_temp=JANELA_TEMP, n_val_meses=N_VAL_MESES,
                  min_treino_temp=MIN_TREINO_TEMP, features=features,
                  chave_emissor="coluna emissor" if "emissor" in painel.columns else "ticker[:4] (heuristica)",
                  sha256_painel=hash_arquivo(ARQ_PAINEL), lightgbm=lgb.__version__, pandas=pd.__version__)
    run_id = hashlib.sha256(json.dumps(config, sort_keys=True, default=str).encode()).hexdigest()[:10]
    run_dir = SAIDA / f"run_{run_id}"
    (run_dir / "modelos").mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps(config, indent=2, default=str), encoding="utf-8")
    print("run:", run_dir.name)

    cols = list(dict.fromkeys(obrig + features + (["emissor"] if "emissor" in painel.columns else [])))
    painel = painel[cols]
    grupos = {pd.Timestamp(m): g for m, g in painel.groupby("data")}
    n_val = {m: {al: len(filtra_alvo(g, al)) for al in ALVOS} for m, g in grupos.items()}
    for al in ALVOS:
        print(f"meses com >= {MIN_OBS} obs validas em {al}: {sum(v[al] >= MIN_OBS for v in n_val.values())}")
    if a.modo == "cv":
        meses = [m for m, v in n_val.items() if max(v.values()) >= MIN_OBS]
    else:
        meses = sorted(grupos)
    meses = sorted(meses)
    if not meses:
        print(f"Nenhum mes elegivel (MIN_OBS={MIN_OBS}). Rode diagnostico_painel.py.")
        return
    if a.n_meses > 0:
        meses = meses[:a.n_meses]
        print(f"AVISO: apenas os {len(meses)} primeiros meses (teste de execucao) -- nao conclua desempenho.")
    print("meses:", meses[0].date(), "->", meses[-1].date(), f"({len(meses)})")
    salvar = not a.sem_modelos
    if a.modo == "cv":
        tarefas = (delayed(roda_mes_cv)(grupos[m], features, m, params, run_dir, salvar) for m in meses)
    else:
        datas = sorted(grupos)
        def janela(m):
            i = datas.index(m)
            ini = datas[max(0, i - JANELA_TEMP)]
            return painel[(painel.data >= ini) & (painel.data <= m)]
        tarefas = (delayed(roda_mes_temporal)(janela(m), features, m, params, run_dir, salvar) for m in meses)
    status = Parallel(n_jobs=a.jobs)(tarefas)
    print("status por mes:", dict(Counter(status)))
    consolida(run_dir)
if __name__ == "__main__":
    main()